# Task 3: Dropout heatmaps and ImageNet ResNet34 components

Add only the official 2026task3 data. Enable GPU and Internet for ImageNet initialization.
Training outputs are saved under /kaggle/working/task3. Collect the two final training outputs into one directory before running the fixed coordinate blends described in the README.

In [ ]:
%pip install -q timm pandas Pillow scikit-learn opencv-python-headless


In [ ]:
from pathlib import Path
Path("/kaggle/working/task3_source").mkdir(exist_ok=True)


In [ ]:
%%writefile /kaggle/working/task3_source/data.py
"""Official-data loading, retina geometry, pyramid rendering and label-synchronised augmentation."""
import math
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2

cv2.setNumThreads(1)

FROZEN_FOLDS = dict(s.split(':') for s in (
    '0001:1,0002:1,0003:3,0004:0,0005:0,0006:3,0007:2,0008:1,0009:4,0010:3,0011:2,0012:1,0013:1,0014:1,0015:0,'
    '0016:4,0017:2,0018:0,0019:3,0020:4,0021:0,0022:4,0023:2,0024:4,0025:0,0026:2,0027:4,0028:0,0029:3,0030:0,'
    '0031:0,0032:4,0033:1,0034:2,0035:0,0036:2,0037:1,0038:3,0039:2,0040:0,0041:2,0042:1,0043:4,0044:2,0045:3,'
    '0046:0,0047:3,0048:1,0049:2,0050:4,0051:1,0052:2,0053:3,0054:2,0055:2,0056:1,0057:0,0058:0,0059:2,0060:4,'
    '0061:3,0062:3,0063:4,0064:4,0065:1,0066:4,0067:3,0068:3,0069:1,0070:4,0071:0,0072:4,0073:4,0074:3,0075:1,'
    '0076:2,0077:2,0078:3,0079:1,0080:3').split(','))

SPLIT2_FOLDS = dict(s.split(':') for s in (
    '0001:0,0002:4,0003:1,0004:2,0005:4,0006:1,0007:4,0008:4,0009:4,0010:1,0011:1,0012:3,0013:2,0014:3,0015:4,'
    '0016:2,0017:3,0018:4,0019:2,0020:2,0021:2,0022:1,0023:3,0024:0,0025:0,0026:4,0027:1,0028:3,0029:3,0030:1,'
    '0031:3,0032:3,0033:2,0034:1,0035:1,0036:0,0037:2,0038:2,0039:0,0040:1,0041:3,0042:1,0043:4,0044:4,0045:3,'
    '0046:2,0047:3,0048:0,0049:2,0050:0,0051:1,0052:4,0053:4,0054:3,0055:2,0056:1,0057:2,0058:0,0059:1,0060:0,'
    '0061:0,0062:2,0063:4,0064:4,0065:4,0066:0,0067:3,0068:3,0069:3,0070:1,0071:0,0072:1,0073:2,0074:0,0075:4,'
    '0076:0,0077:0,0078:2,0079:0,0080:3').split(','))

def find_data_dir(hint=None):
    roots = [Path(hint)] if hint else [Path('/kaggle/input')]
    for root in roots:
        hits = sorted(root.rglob('fovea_localization_train_GT.csv')) if root.exists() else []
        if hits:
            return hits[0].parent
    raise FileNotFoundError('Attach the official 2026task3 competition data')

def stem_key(name):
    s = Path(str(name)).stem
    return str(int(s)) if s.isdigit() else s

def retina_circle(rgb):
    """Centre and radius of the bright field of view (largest connected component), original pixels."""
    h, w = rgb.shape[:2]
    f = 512 / max(h, w)
    small = cv2.resize(rgb, (max(1, round(w * f)), max(1, round(h * f))), interpolation=cv2.INTER_AREA)
    n, _, stats, _ = cv2.connectedComponentsWithStats((small.max(2) > 20).astype(np.uint8), 8)
    x, y, bw, bh = stats[1 + np.argmax(stats[1:, 4]), :4] if n > 1 else (0, 0, small.shape[1], small.shape[0])
    sx, sy = w / small.shape[1], h / small.shape[0]
    return (x + bw / 2) * sx, (y + bh / 2) * sy, max(bw * sx, bh * sy) / 2

class Fundus:
    """One image kept as an anti-aliased pyramid (1/2, 1/4, 1/8) plus its field-of-view circle."""

    def __init__(self, name, rgb, xy=None, fold=-1):
        self.name, self.fold = name, fold
        self.group = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
        self.h, self.w = rgb.shape[:2]
        self.xy = None if xy is None else np.asarray(xy, float)
        self.cx, self.cy, self.r = retina_circle(rgb)
        self.levels, self.to_level, src = {}, {}, rgb
        for f in (2, 4, 8):
            lw, lh = max(1, round(self.w / f)), max(1, round(self.h / f))
            src = cv2.resize(src, (lw, lh), interpolation=cv2.INTER_AREA)
            fx, fy = self.w / lw, self.h / lh
            self.levels[f] = src

            self.to_level[f] = np.array([[1 / fx, 0, .5 / fx - .5], [0, 1 / fy, .5 / fy - .5], [0, 0, 1.]])

def _read(path):
    return cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def load_dataset(data_dir, split=1):
    if split not in (1, 2):
        raise ValueError("Choose grouped split 1 or 2")
    folds = FROZEN_FOLDS if split == 1 else SPLIT2_FOLDS
    data_dir = Path(data_dir)
    labels = pd.read_csv(data_dir / 'fovea_localization_train_GT.csv', dtype={'data': str})
    sample = pd.read_csv(data_dir / 'sample_submission.csv')
    files = {stem_key(p.name): p for p in (data_dir / 'train').glob('*.jpg')}
    rows = [(files[stem_key(r.data)], (r.Fovea_X, r.Fovea_Y)) for r in labels.itertuples()]
    tests = sorted((data_dir / 'test').glob('*.jpg'))
    with ThreadPoolExecutor(8) as pool:
        train = list(pool.map(lambda r: Fundus(r[0].name, _read(r[0]), r[1], int(folds[r[0].stem])), rows))
        test = list(pool.map(lambda p: Fundus(p.name, _read(p)), tests))
    assert len(train) == 80 and len(test) == 20 and len(sample) == 40 and list(sample) == ['ImageID', 'value']
    assert all(np.isfinite(fd.xy).all() and 0 <= fd.xy[0] < fd.w and 0 <= fd.xy[1] < fd.h for fd in train)
    for fold in range(5):
        if not {fd.group for fd in train if fd.fold == fold}.isdisjoint(
                {fd.group for fd in train if fd.fold != fold}):
            raise ValueError("Identical training images cross validation folds")
    return train, test, sample

def render(fd, M, size):
    """Warp from the pyramid level just above the needed resolution; M maps original pixels to input pixels."""
    scale = math.sqrt(abs(np.linalg.det(M[:2, :2])))
    ok = [f for f in fd.levels if 1 / f >= .98 * scale]
    f = max(ok) if ok else min(fd.levels)
    A = M @ np.linalg.inv(fd.to_level[f])
    return cv2.warpAffine(fd.levels[f], A[:2], (size, size), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def square(cx, cy, side, size):
    """Isotropic map of a side x side window centred on (cx, cy) to a size x size input."""
    s = size / side
    return np.array([[s, 0, (size - 1) / 2 - cx * s], [0, s, (size - 1) / 2 - cy * s], [0, 0, 1.]])

def coarse_base(fd, size, margin=.03, crop='fov'):
    """'fov': isotropic square around the field-of-view circle; 'image': whole image squeezed to size x size."""
    if crop == 'image':
        sx, sy = size / fd.w, size / fd.h
        return np.array([[sx, 0, .5 * sx - .5], [0, sy, .5 * sy - .5], [0, 0, 1.]])
    return square(fd.cx, fd.cy, 2 * fd.r * (1 + margin), size)

def hflip(size):
    return np.array([[-1, 0, size - 1], [0, 1, 0], [0, 0, 1.]])

def apply(M, xy):
    p = M @ np.array([xy[0], xy[1], 1.])
    return p[:2] / p[2]

def random_affine(size, rng, rot, scale, shift, flip=True):
    c = (size - 1) / 2
    s = math.exp(rng.uniform(-math.log(scale), math.log(scale)))
    A = np.vstack([cv2.getRotationMatrix2D((c, c), rng.uniform(-rot, rot), s), [0, 0, 1.]])
    A[:2, 2] += rng.uniform(-shift, shift, 2) * size
    return hflip(size) @ A if flip and rng.random() < .5 else A

def photometric(img, rng, p=1.):
    """Colour, contrast, gamma, saturation, blur and noise jitter inside the retina; labels untouched."""
    x = img.astype(np.float32) / 255
    if p <= 0:
        return x
    inside = (img.max(2, keepdims=True) > 10).astype(np.float32)
    x = x * rng.uniform(1 - .1 * p, 1 + .1 * p, 3).astype(np.float32)
    x = (x - .5) * rng.uniform(1 - .25 * p, 1 + .25 * p) + .5 + rng.uniform(-.1 * p, .1 * p)
    x = np.clip(x, 0, 1) ** rng.uniform(1 - .2 * p, 1 + .25 * p)
    g = x.mean(2, keepdims=True)
    x = g + (x - g) * rng.uniform(1 - .3 * p, 1 + .3 * p)
    if rng.random() < .3 * p:
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(.3, 1.2))
    if rng.random() < .3 * p:
        x = x + rng.normal(0, .02 * p, x.shape).astype(np.float32)
    return (np.clip(x, 0, 1) * inside).astype(np.float32)

def random_erase(img, rng, p, frac=.25):
    """Blank one random rectangle (any position, the fovea included) so context must also be used."""
    if p <= 0 or rng.random() >= p:
        return img
    s = img.shape[0]
    h, w = (rng.uniform(.08, frac, 2) * s).astype(int)
    y, x = rng.integers(0, s - h), rng.integers(0, s - w)
    img = img.copy()
    img[y:y + h, x:x + w] = img[img.max(2) > 10].mean(0) if (img.max(2) > 10).any() else 0
    return img

def coarse_sample(fd, size, rng, aug):
    M = coarse_base(fd, size, crop=aug.get('crop', 'fov'))
    if aug['geo']:
        M = random_affine(size, rng, aug['rot'], aug['scale'], aug['shift'], aug.get('flip', True)) @ M
    img = random_erase(render(fd, M, size), rng, aug.get('erase', 0.))
    return photometric(img, rng, aug['photo']), apply(M, fd.xy)

def fine_sample(fd, size, crop, rng, aug):
    """Window around the label plus a random offset wider than typical coarse errors (avoids a centre bias)."""
    jitter = aug['jitter_px'] if 'jitter_px' in aug else aug['jitter'] * crop
    wide = aug['wide_px'] if 'wide_px' in aug else aug['wide'] * crop
    for _ in range(50):
        d = rng.normal(0, jitter, 2) if rng.random() < aug.get('p_narrow', .7) else rng.uniform(-wide, wide, 2)
        M = square(fd.xy[0] + d[0], fd.xy[1] + d[1], crop, size)
        if aug['geo']:
            M = random_affine(size, rng, aug['rot'], aug['scale'], 0, aug.get('flip', True)) @ M
        t = apply(M, fd.xy)
        if (t > .08 * size).all() and (t < .92 * size).all():
            break
    return photometric(render(fd, M, size), rng, aug['photo']), t


def dataset_fingerprint(root):
    """Bind every pipeline's outputs to the same official image files and label/template CSVs."""
    root = Path(root)
    digest = hashlib.sha256()
    files = [root / 'fovea_localization_train_GT.csv', root / 'sample_submission.csv']
    files += sorted((root / 'train').glob('*.jpg')) + sorted((root / 'test').glob('*.jpg'))
    for path in files:
        digest.update(str(path.relative_to(root)).encode())
        digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


In [ ]:
%%writefile /kaggle/working/task3_source/submission.py
"""Official 20-image, 40-row coordinate submission format."""
import re
import numpy as np
from data import stem_key


def write_submission(test, pred, sample, path):
    pred = np.asarray(pred)
    if len(test) != 20 or pred.shape != (20, 2) or not np.isfinite(pred).all():
        raise ValueError('Expected twenty finite (x, y) predictions')
    mapping = {stem_key(fd.name): point for fd, point in zip(test, pred)}
    if len(mapping) != 20 or list(sample) != ['ImageID', 'value'] or len(sample) != 40:
        raise ValueError('Invalid official image names or submission template')
    for fd, point in zip(test, pred):
        if not (0 <= point[0] < fd.w and 0 <= point[1] < fd.h):
            raise ValueError(f'Coordinate outside the original image: {fd.name}')
    values, seen = [], set()
    for image_id in sample.ImageID:
        match = re.fullmatch(r'(.+)_Fovea_([XY])', str(image_id))
        if match is None:
            raise ValueError('Invalid coordinate ID in submission template')
        name, axis = stem_key(match[1]), match[2]
        if name not in mapping or (name, axis) in seen:
            raise ValueError('Unknown or repeated coordinate ID')
        seen.add((name, axis))
        values.append(float(mapping[name][0 if axis == 'X' else 1]))
    if len(seen) != 40:
        raise ValueError('Incomplete coordinate submission')
    frame = sample.copy()
    frame['value'] = values
    frame.to_csv(path, index=False)
    return frame


In [ ]:
%%writefile /kaggle/working/task3_source/classic.py
"""Scratch heatmap/regression and ImageNet ResNet component training; exact original affine/decode protocol."""
from pathlib import Path
import copy, hashlib, json, math, os, random, re, time, zipfile
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold

from torchvision.models import resnet18, resnet34, ResNet18_Weights, ResNet34_Weights
from torchvision.ops.misc import FrozenBatchNorm2d
from data import FROZEN_FOLDS, SPLIT2_FOLDS

SEED = 2026
PIPELINE_VERSION = 'fovea-component-training'
MODES = ('global', 'sharp', 'local9')
cv2.setNumThreads(0)
torch.set_num_threads(2)
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rmse(target, pred):
    return float(np.sqrt(np.mean((np.asarray(target)-np.asarray(pred))**2)))

def key(value):
    stem = Path(str(value)).stem
    return str(int(stem)) if stem.isdigit() else stem

def apply_point(xy, M):
    point = np.r_[xy, 1.] @ M.T
    return point[:2] / point[2]

def load_official(root, split=1):
    paths = list(Path(root).rglob('fovea_localization_train_GT.csv'))
    assert len(paths)==1, 'Mount exactly the official 2026task3 competition data'
    base=paths[0].parent
    labels=pd.read_csv(paths[0],dtype={'data':str})
    assert {'data','Fovea_X','Fovea_Y'} <= set(labels)
    lookup={key(p.name):p for p in (base/'train').glob('*.jpg')}
    records=[]; pixels={}
    for r in labels.to_dict('records'):
        path=lookup[key(r['data'])]
        rgb=np.array(Image.open(path).convert('RGB'))
        h,w=rgb.shape[:2]; xy=np.array([r['Fovea_X'],r['Fovea_Y']],dtype=float)
        assert np.isfinite(xy).all() and ((xy>=0)&(xy<[w,h])).all()
        digest=hashlib.sha256(str(rgb.shape).encode()+rgb.tobytes()).hexdigest()

        cw=round(w*min(1,768/max(w,h))); ch=round(h*min(1,768/max(w,h)))
        small=cv2.resize(rgb,(cw,ch),interpolation=cv2.INTER_AREA)

        mask=(small.max(2)>20).astype('uint8')
        n,comp,stats,_=cv2.connectedComponentsWithStats(mask,8)
        if n>1:
            x,y,bw,bh,area=stats[1+np.argmax(stats[1:,4])]
            margin=round(.015*max(cw,ch)); x0=max(0,x-margin); y0=max(0,y-margin)
            x1=min(cw,x+bw+margin); y1=min(ch,y+bh+margin)
        else: x0,y0,x1,y1=0,0,cw,ch
        pixels[path.name]=(small,(x0,y0,x1,y1))
        records.append(dict(image=path.name,path=str(path),width=w,height=h,cw=cw,ch=ch,
                            x=xy[0],y=xy[1],visible=bool(xy.any()),group=digest))
    df=pd.DataFrame(records)
    assert len(df)==80 and len(lookup)==80 and len(list((base/'test').glob('*.jpg')))==20
    assert df.image.nunique()==80

    if split not in (1, 2):
        raise ValueError('Choose grouped split 1 or 2')
    assignment = FROZEN_FOLDS if split == 1 else SPLIT2_FOLDS
    df['fold'] = [int(assignment[Path(name).stem]) for name in df.image]
    for fold in range(5):
        assert set(df[df.fold!=fold].group).isdisjoint(df[df.fold==fold].group)
    assert (df.fold>=0).all()
    sample=pd.read_csv(base/'sample_submission.csv')
    assert list(sample)==['ImageID','value'] and len(sample)==40
    return base,df,pixels,sample

class Fundus(Dataset):
    def __init__(self,df,pixels,cfg,train=False):
        self.df=df.reset_index(drop=True); self.pixels=pixels; self.cfg=cfg; self.train=train
    def __len__(self): return len(self.df)*(self.cfg['repeat'] if self.train else 1)
    def __getitem__(self,i):
        r=self.df.iloc[i%len(self.df)]; image,box=self.pixels[r.image]
        h,w=image.shape[:2]; s=self.cfg['size']
        x0,y0,x1,y1=box if self.cfg['crop'] else (0,0,w,h)
        M=np.array([[s/(x1-x0),0,-x0*s/(x1-x0)],[0,s/(y1-y0),-y0*s/(y1-y0)],[0,0,1.]])
        original=np.array([r.x/r.width*w,r.y/r.height*h])
        if self.train and self.cfg['geometry']:
            a=cv2.getRotationMatrix2D((s/2,s/2),np.random.uniform(-15,15),np.random.uniform(.9,1.1))
            a[:,2]+=np.random.uniform(-.04,.04,2)*s
            A=np.vstack([a,[0,0,1]])
            if np.random.rand()<.5: A=np.array([[-1,0,s-1],[0,1,0],[0,0,1.]])@A
            trial=A@M; pt=apply_point(original,trial)
            if not r.visible or ((pt>=0)&(pt<s)).all(): M=trial
        if self.cfg['clahe']:
            lab=cv2.cvtColor(image,cv2.COLOR_RGB2LAB)
            lab[:,:,0]=cv2.createCLAHE(2.,(8,8)).apply(lab[:,:,0])
            image=cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)
        image=cv2.warpAffine(image,M[:2],(s,s),flags=cv2.INTER_LINEAR).astype('float32')/255
        if self.train and self.cfg['photo']:
            image=np.clip((image-.5)*np.random.uniform(.8,1.2)+.5+np.random.uniform(-.08,.08),0,1)
            image=np.power(image,np.random.uniform(.85,1.15)).astype('float32')
        xy=apply_point(original,M)/s if r.visible else np.array([0.,0.])

        M_original=M@np.diag([w/r.width,h/r.height,1.])
        return (torch.from_numpy(image.transpose(2,0,1).copy()),torch.tensor(xy,dtype=torch.float32),
                torch.tensor(r.visible,dtype=torch.float32),torch.tensor(np.linalg.inv(M_original),dtype=torch.float32))

def block(a,b):
    return nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.GroupNorm(4,b),nn.SiLU(),
                         nn.Conv2d(b,b,3,padding=1,bias=False),nn.GroupNorm(4,b),nn.SiLU())

class ScratchLocator(nn.Module):
    def __init__(self,width=24,model='heatmap'):
        super().__init__(); self.model=model
        self.e1=block(3,width); self.e2=block(width,width*2); self.e3=block(width*2,width*4)
        self.e4=block(width*4,width*8); self.center=block(width*8,width*8)
        self.visibility=nn.Linear(width*8,1)
        if model=='heatmap':
            self.d3=block(width*12,width*4); self.d2=block(width*6,width*2)
            self.head=nn.Conv2d(width*2,1,1)
        else:
            self.reg=nn.Sequential(nn.AdaptiveAvgPool2d((4,4)),nn.Flatten(),nn.Linear(width*8*16,128),nn.SiLU(),nn.Linear(128,2))
    def forward(self,x):
        a=self.e1(F.avg_pool2d(x,2)); b=self.e2(F.avg_pool2d(a,2)); c=self.e3(F.avg_pool2d(b,2))
        d=self.e4(F.avg_pool2d(c,2)); z=self.center(F.avg_pool2d(d,2))
        vis=self.visibility(z.mean((2,3))).squeeze(1)
        if self.model!='heatmap': return self.reg(z).sigmoid(),vis,None
        u=self.d3(torch.cat([F.interpolate(z,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.d2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        logits=self.head(u).squeeze(1); h,w=logits.shape[-2:]
        prob=logits.flatten(1).softmax(1).view(-1,h,w)
        gx=(torch.arange(w,device=x.device,dtype=x.dtype)+.5)/w
        gy=(torch.arange(h,device=x.device,dtype=x.dtype)+.5)/h
        xy=torch.stack([(prob.sum(1)*gx).sum(1),(prob.sum(2)*gy).sum(1)],1)
        return xy,vis,logits

def loss_fn(output,xy,visible,cfg):
    pred,vis,logits=output
    coordinate=((pred-xy)**2).sum(1)
    loss=(coordinate*visible).sum()/visible.sum().clamp_min(1)*20
    if logits is not None:
        h,w=logits.shape[-2:]; dtype=logits.dtype; device=logits.device
        gx=(torch.arange(w,device=device,dtype=dtype)+.5)/w
        gy=(torch.arange(h,device=device,dtype=dtype)+.5)/h
        sq=(gx[None,None,:]-xy[:,0,None,None])**2+(gy[None,:,None]-xy[:,1,None,None])**2
        target=torch.exp(-sq/(2*(cfg['sigma']/cfg['size'])**2)).flatten(1)
        target=target/target.sum(1,keepdim=True).clamp_min(1e-20)
        ce=-(target*logits.flatten(1).log_softmax(1)).sum(1)
        loss=loss+(ce*visible).sum()/visible.sum().clamp_min(1)
    return loss+F.binary_cross_entropy_with_logits(vis,visible)*.1

@torch.inference_mode()
def predict(model,loader,device,size,tta=False):
    model.eval(); preds=[]; visibilities=[]
    for image,xy,visible,inverse in loader:
        image=image.to(device); xyhat,vis,_=model(image); vs=vis.sigmoid()
        if tta:
            xflip,vflip,_=model(image.flip(-1)); xflip[:,0]=1-1/size-xflip[:,0]
            xyhat=(xyhat+xflip)/2; vs=(vs+vflip.sigmoid())/2
        netxy=xyhat.cpu()*size
        hom=torch.cat([netxy,torch.ones(len(netxy),1)],1)
        raw=torch.bmm(inverse,hom.unsqueeze(-1)).squeeze(-1)
        raw=raw[:,:2]/raw[:,2:3]

        preds.append(raw.numpy()); visibilities.append(vs.cpu().numpy())
    return np.concatenate(preds),np.concatenate(visibilities)

def checkpoint_fingerprint(cfg,fold,df):
    identity=dict(pipeline=PIPELINE_VERSION,cfg=cfg,fold=fold,seed=SEED,
        images=df[['image','group','x','y','fold']].to_dict('records'))
    if cfg['accumulate']>1:
        identity['accumulation_rule']='sample_weighted_partial_group'
    return hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()

def test_pixels(base):
    pixels={}; rows=[]
    for path in sorted((base/'test').glob('*.jpg')):

        rgb=np.array(Image.open(path).convert('RGB')); h,w=rgb.shape[:2]
        cw=round(w*min(1,768/max(w,h))); ch=round(h*min(1,768/max(w,h)))
        im=cv2.resize(rgb,(cw,ch),interpolation=cv2.INTER_AREA)
        n,_,stats,_=cv2.connectedComponentsWithStats((im.max(2)>20).astype('uint8'),8)
        if n>1:
            x,y,bw,bh,_=stats[1+np.argmax(stats[1:,4])]; m=round(.015*max(cw,ch))
            box=(max(0,x-m),max(0,y-m),min(cw,x+bw+m),min(ch,y+bh+m))
        else: box=(0,0,cw,ch)
        pixels[path.name]=(im,box)
        rows.append(dict(image=path.name,width=w,height=h,cw=cw,ch=ch,x=0.,y=0.,visible=False))
    return pd.DataFrame(rows),pixels
def decode_heatmap(logits,mode):
    h,w=logits.shape[-2:]
    z=logits.flatten(1)
    if mode=='sharp':z=z/.5
    elif mode=='local9':
        peak=z.argmax(1); py=peak//w;px=peak%w
        yy=torch.arange(h,device=z.device)[None,:,None];xx=torch.arange(w,device=z.device)[None,None,:]
        mask=((yy-py[:,None,None]).abs()<=4)&((xx-px[:,None,None]).abs()<=4)
        z=z.masked_fill(~mask.flatten(1),-torch.inf)
    else:assert mode=='global'
    prob=z.softmax(1).reshape(-1,h,w)
    gx=(torch.arange(w,device=z.device,dtype=z.dtype)+.5)/w
    gy=(torch.arange(h,device=z.device,dtype=z.dtype)+.5)/h
    return torch.stack([(prob.sum(1)*gx).sum(1),(prob.sum(2)*gy).sum(1)],1)

@torch.inference_mode()
def predict_decoders(model,loader,device,size):
    model.eval(); result={mode:[] for mode in MODES}
    for image,_,_,inverse in loader:
        _,_,logits=model(image.to(device));assert logits is not None
        for mode in MODES:
            xy=decode_heatmap(logits,mode).cpu()*size
            hom=torch.cat([xy,torch.ones(len(xy),1)],1)
            raw=torch.bmm(inverse,hom.unsqueeze(-1)).squeeze(-1)
            result[mode].append((raw[:,:2]/raw[:,2:3]).numpy())
    return {mode:np.concatenate(value) for mode,value in result.items()}
def train_fold(name,cfg,fold,df,pixels,out,device):
    dest=out/name/f'fold{fold}'; dest.mkdir(parents=True,exist_ok=True)
    fingerprint=checkpoint_fingerprint(cfg,fold,df)
    if (dest/'complete.json').exists():
        old=json.loads((dest/'complete.json').read_text()); assert old['fingerprint']==fingerprint
        return
    seed_all(SEED+fold+cfg.get("seed_offset",0))
    tr=df[df.fold!=fold]; va=df[df.fold==fold]
    train=DataLoader(Fundus(tr,pixels,cfg,True),batch_size=cfg['batch'],shuffle=True,num_workers=0)
    valid=DataLoader(Fundus(va,pixels,cfg),batch_size=cfg['batch'],num_workers=0)
    model=Locator(cfg['width'],cfg['model']).to(device)
    if hasattr(model,'encoder'):
        encoder_ids={id(p) for p in model.encoder.parameters()}
        groups=[dict(params=model.encoder.parameters(),lr=cfg['encoder_lr']),
                dict(params=[p for p in model.parameters() if id(p) not in encoder_ids],lr=cfg['lr'])]
        opt=torch.optim.AdamW(groups,weight_decay=cfg['weight_decay'])
        sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda t:.01+.99*.5*(1+math.cos(math.pi*t/cfg['epochs'])))
    else:
        opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
        sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,cfg['epochs'],eta_min=cfg['lr']*.01)
    history=[]; start=0; average=None; n_average=0

    if (dest/'last.pt').exists():
        state=torch.load(dest/'last.pt',map_location='cpu',weights_only=False)
        assert state['fingerprint']==fingerprint
        model.load_state_dict(state['model']); opt.load_state_dict(state['opt']); sched.load_state_dict(state['sched'])
        start=state['epoch']+1; history=state['history']; average=state['average']; n_average=state['n_average']
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng']); torch.set_rng_state(state['torch_rng'])
        if device.type=='cuda': torch.cuda.set_rng_state_all(state['cuda_rng'])
    for epoch in range(start,cfg['epochs']):
        begin=time.time(); model.train(); total=0.; count=0
        opt.zero_grad(set_to_none=True)
        for step,(image,xy,visible,_) in enumerate(train):
            image=image.to(device); xy=xy.to(device); visible=visible.to(device)
            loss=loss_fn(model(image),xy,visible,cfg)
            assert torch.isfinite(loss),f'Nonfinite loss in {name} fold {fold}'
            group_start=(step//cfg['accumulate'])*cfg['batch']*cfg['accumulate']
            group_samples=min(cfg['batch']*cfg['accumulate'],len(train.dataset)-group_start)
            (loss*len(image)/group_samples).backward()
            if (step+1)%cfg['accumulate']==0 or step+1==len(train):
                nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step(); opt.zero_grad(set_to_none=True)
            total+=float(loss.detach())*len(image); count+=len(image)
        sched.step()

        if epoch>=cfg['epochs']-10:
            weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            if average is None: average=weights
            else:
                for k in average: average[k].add_((weights[k]-average[k])/(n_average+1))
            n_average+=1
        record=dict(epoch=epoch+1,train_loss=total/count,lr=opt.param_groups[0]['lr'],seconds=time.time()-begin)
        if (epoch+1)%5==0 or epoch==0:
            p,_=predict(model,valid,device,cfg['size'])
            record['val_rmse']=rmse(va[['x','y']].values,p)
        history.append(record)
        state=dict(fingerprint=fingerprint,model={k:v.detach().cpu() for k,v in model.state_dict().items()},opt=opt.state_dict(),sched=sched.state_dict(),
                   epoch=epoch,history=history,average=average,n_average=n_average,
                   python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
                   cuda_rng=torch.cuda.get_rng_state_all() if device.type=='cuda' else [])
        torch.save(state,dest/'last.tmp'); os.replace(dest/'last.tmp',dest/'last.pt')
        pd.DataFrame(history).to_csv(dest/'history.csv',index=False)
        if 'val_rmse' in record: print(name,fold,record,flush=True)
    assert n_average==10
    model.load_state_dict(average)
    p,v=predict(model,valid,device,cfg['size']); pt,vt=predict(model,valid,device,cfg['size'],True)
    result=va[['image','fold','x','y','group']].copy()
    result['pred_x']=p[:,0]; result['pred_y']=p[:,1]; result['tta_x']=pt[:,0]; result['tta_y']=pt[:,1]
    result['visibility']=v; result.to_csv(dest/'oof.csv',index=False)
    torch.save(dict(model=average,cfg=cfg,fingerprint=fingerprint,fold=fold),dest/'model.pt')
    summary=dict(fingerprint=fingerprint,rmse=rmse(va[['x','y']],p),tta_rmse=rmse(va[['x','y']],pt),
                 cfg=cfg,fold=fold,seconds=float(sum(r['seconds'] for r in history)))
    (dest/'complete.json').write_text(json.dumps(summary,indent=2))
    print('COMPLETE',name,fold,summary,flush=True)
    del model,opt,state; torch.cuda.empty_cache()

def freeze_batch_norm(module):
    for name,child in list(module.named_children()):
        if isinstance(child,nn.BatchNorm2d):
            frozen=FrozenBatchNorm2d(child.num_features,eps=child.eps)
            with torch.no_grad():
                for attr in ['weight','bias','running_mean','running_var']:
                    getattr(frozen,attr).copy_(getattr(child,attr))
            setattr(module,name,frozen)
        else:freeze_batch_norm(child)

class ResNetLocator(nn.Module):
    def __init__(self,model):
        super().__init__();self.model='heatmap'
        backbone='resnet34' if '34' in model else 'resnet18'
        self.encoder=(resnet34 if backbone=='resnet34' else resnet18)(weights=None)
        if 'imagenet' in model:
            weights = ResNet34_Weights.IMAGENET1K_V1 if backbone == 'resnet34' else ResNet18_Weights.IMAGENET1K_V1
            self.encoder.load_state_dict(weights.get_state_dict(progress=True, check_hash=True))
        self.encoder.fc=nn.Identity();freeze_batch_norm(self.encoder)
        self.d4=block(768,128);self.d3=block(256,64);self.d2=block(128,32)
        self.head=nn.Conv2d(32,1,1);self.visibility=nn.Linear(512,1)
        self.register_buffer('rgb_mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('rgb_std',torch.tensor([.229,.224,.225]).view(1,3,1,1))
    def forward(self,x):
        x=(x-self.rgb_mean)/self.rgb_std;e=self.encoder
        x=e.maxpool(e.relu(e.bn1(e.conv1(x))))
        a=e.layer1(x);b=e.layer2(a);c=e.layer3(b);z=e.layer4(c)
        u=self.d4(torch.cat([F.interpolate(z,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.d3(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.d2(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        logits=self.head(u).squeeze(1)
        return decode_heatmap(logits,'global'),self.visibility(z.mean((2,3))).squeeze(1),logits

def Locator(width=24, model='heatmap'):
    if model.startswith('resnet'):
        return ResNetLocator(model)
    if model == 'heatmap_dropout':
        net = ScratchLocator(width, 'heatmap')
        for name in ['e1', 'e2', 'e3', 'e4', 'center', 'd3', 'd2']:
            getattr(net, name).add_module('dropout', nn.Dropout2d(.1))
        return net
    return ScratchLocator(width, model)


In [ ]:
%%writefile /kaggle/working/task3_source/train_components.py
"""Train the scratch and torchvision ResNet members and their completed controlled comparisons."""
import argparse
import copy
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

import classic
from submission import write_submission
from data import load_dataset, find_data_dir, dataset_fingerprint

DEFAULT = dict(model='heatmap', width=24, size=256, lr=1e-3, weight_decay=1e-4,
               crop=True, geometry=True, photo=True, clahe=False, sigma=8., epochs=100,
               repeat=2, batch=8, accumulate=1)
SCRATCH = dict(DEFAULT, model='heatmap_dropout', lr=3e-4, epochs=200)
TRANSFER = dict(DEFAULT, model='resnet34_imagenet', lr=1e-3, encoder_lr=1e-4, epochs=100)
COMPONENTS = {'dropout_split1': (SCRATCH, 1), 'dropout_split2': (SCRATCH, 2),
              'resnet34_transfer': (TRANSFER, 1)}
BASELINE = {
    'heatmap_main': DEFAULT,
    'coordinate_regression': dict(DEFAULT, model='regression'),
    'without_geometry': dict(DEFAULT, geometry=False),
    'without_photometric': dict(DEFAULT, photo=False),
    'without_retina_crop': dict(DEFAULT, crop=False),
    'heatmap_lr3e4': dict(DEFAULT, lr=3e-4),
    'gaussian_sigma4': dict(DEFAULT, sigma=4.),
    'with_clahe': dict(DEFAULT, clahe=True),
    'input384': dict(DEFAULT, size=384, sigma=12., batch=4, accumulate=2),
}
DURATION = {'heatmap_lr3e4': dict(DEFAULT, lr=3e-4),
            'duration_200': dict(DEFAULT, lr=3e-4, epochs=200),
            'dropout_100': dict(DEFAULT, model='heatmap_dropout', lr=3e-4),
            'dropout_split1': SCRATCH}
TRANSFER_COMPARISON = {
    'resnet18_scratch': dict(DEFAULT, model='resnet18_scratch', lr=1e-3, encoder_lr=3e-4),
    'resnet18_imagenet_lr3e4': dict(DEFAULT, model='resnet18_imagenet', lr=1e-3, encoder_lr=3e-4),
    'resnet18_imagenet_lr1e4': dict(DEFAULT, model='resnet18_imagenet', lr=1e-3, encoder_lr=1e-4),
    'resnet34_transfer': TRANSFER,
}
EXPERIMENTS = {**{name: (cfg, 1) for name, cfg in BASELINE.items()},
               **{name: (cfg, 1) for name, cfg in DURATION.items()},
               **{name: (cfg, 1) for name, cfg in TRANSFER_COMPARISON.items()}}


def export_component(name, cfg, split, base, frame, pixels, sample, root, device):
    directory = root / name
    test_frame, test_pixels = classic.test_pixels(base)
    oof = {mode: np.full((len(frame), 2), np.nan) for mode in classic.MODES}
    tests = {mode: [] for mode in classic.MODES}
    for fold in range(5):
        checkpoint = directory / f'fold{fold}' / 'model.pt'
        state = torch.load(checkpoint, map_location='cpu', weights_only=False)
        if state['cfg'] != cfg or state['fingerprint'] != classic.checkpoint_fingerprint(cfg, fold, frame):
            raise ValueError('Checkpoint data, configuration or grouped split differs')
        # Full model state replaces initialization; inference does not download ImageNet weights.
        model_name = cfg['model'].replace('_imagenet', '_scratch')
        model = classic.Locator(cfg['width'], model_name).to(device)
        model.load_state_dict(state['model'])
        valid = frame[frame.fold == fold]
        for items, image_cache, is_valid in [(valid, pixels, True), (test_frame, test_pixels, False)]:
            loader = DataLoader(classic.Fundus(items, image_cache, cfg), batch_size=cfg['batch'], num_workers=0)
            if cfg['model'] == 'regression':
                values = {'global': classic.predict(model, loader, device, cfg['size'])[0]}
            else:
                values = classic.predict_decoders(model, loader, device, cfg['size'])
            for mode, predictions in values.items():
                if is_valid:
                    oof[mode][valid.index] = predictions
                else:
                    tests[mode].append(predictions)
        del model
    modes = ['global'] if cfg['model'] == 'regression' else list(classic.MODES)
    metrics = []
    for mode in modes:
        predicted_test = np.mean(tests[mode], axis=0)
        if not np.isfinite(oof[mode]).all() or not np.isfinite(predicted_test).all():
            raise ValueError('Incomplete coordinate predictions')
        np.savez_compressed(directory / f'predictions_{mode}.npz', oof=oof[mode], test=predicted_test,
                            train_names=frame.image.to_numpy(dtype=str), test_names=test_frame.image.to_numpy(dtype=str),
                            target=frame[['x', 'y']].to_numpy(), folds=frame.fold.to_numpy(),
                            groups=frame.group.to_numpy(dtype=str), split=split,
                            dataset_fingerprint=dataset_fingerprint(base))
        metrics.append({'decoder': mode, 'rmse': classic.rmse(frame[['x', 'y']].to_numpy(), oof[mode])})
    pd.DataFrame(metrics).to_csv(directory / 'decoder_metrics.csv', index=False)
    chosen = 'global' if cfg['model'] == 'regression' else 'sharp'
    train, test, _ = load_dataset(base, split)
    if [item.name for item in test] != test_frame.image.tolist():
        raise ValueError('Test prediction order differs from official image order')
    write_submission(test, np.mean(tests[chosen], axis=0), sample, directory / 'submission.csv')
    return metrics


def run_component(name, cfg, split, data, output, device, folds=range(5)):
    cfg = copy.deepcopy(cfg)
    base, frame, pixels, sample = classic.load_official(data, split)
    directory = output / name
    directory.mkdir(parents=True, exist_ok=True)
    plan = {'config': cfg, 'split': split}
    path = directory / 'config.json'
    if path.exists() and json.loads(path.read_text()) != plan:
        raise ValueError('Component configuration changed; use a new output directory')
    path.write_text(json.dumps(plan, indent=2))
    frame.to_csv(directory / 'training_manifest.csv', index=False)
    for fold in folds:
        classic.train_fold(name, cfg, fold, frame, pixels, output, device)
    if all((directory / f'fold{fold}' / 'complete.json').exists() for fold in range(5)):
        return export_component(name, cfg, split, base, frame, pixels, sample, output, device)
    return None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data', default=None)
    parser.add_argument('--out', type=Path, default=Path('outputs'))
    parser.add_argument('--group', choices=['final', 'baseline', 'duration', 'transfer', 'all'], default='final')
    parser.add_argument('--run', choices=sorted(COMPONENTS.keys() | EXPERIMENTS.keys()))
    parser.add_argument('--folds', nargs='+', type=int, choices=range(5), default=list(range(5)))
    parser.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    parser.add_argument('--list', action='store_true')
    args = parser.parse_args(argv)
    if args.list:
        print('\n'.join(sorted(COMPONENTS.keys() | EXPERIMENTS.keys())))
        return
    selection = dict(COMPONENTS) if args.group == 'final' else (
        {name: (cfg, 1) for name, cfg in {'baseline': BASELINE, 'duration': DURATION,
                                        'transfer': TRANSFER_COMPARISON}.get(args.group, {}).items()})
    if args.group == 'all':
        selection = {**COMPONENTS, **EXPERIMENTS}
    if args.run:
        selection = {args.run: {**COMPONENTS, **EXPERIMENTS}[args.run]}
    data = find_data_dir(args.data)
    root = args.out / 'components'
    for name, (config, split) in selection.items():
        print(name, run_component(name, config, split, data, root, torch.device(args.device), args.folds), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
import subprocess
GROUP = 'final'
subprocess.run(["python", "/kaggle/working/task3_source/train_components.py", "--data", "/kaggle/input", "--out", "/kaggle/working/task3", "--group", GROUP], check=True)
